In [17]:
import pandas as pd
import os

print("Files in raw folder:", os.listdir('../data/raw/'))

movies = pd.read_csv('../data/raw/movies.csv')
ratings = pd.read_csv('../data/raw/ratings.csv')

print("\n--- Movies Data ---")
display(movies.head())

print("\n--- Ratings Data ---")
display(ratings.head())

Files in raw folder: ['movies.csv', 'ratings.csv']

--- Movies Data ---


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



--- Ratings Data ---


,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523


In [ ]:
print("Movies shape:", movies.shape)      
print("Ratings shape:", ratings.shape)    

print("\nMovies info:")
movies.info()

print("\nRatings info:")
ratings.info()

print("\nMovies describe:")
display(movies.describe(include='all'))

print("\nRatings describe:")
display(ratings.describe())

Movies shape: (10329, 3)
Ratings shape: (105339, 4)

Movies info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB

Ratings info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     105339 non-null  int64  
 1   movieId    105339 non-null  int64  
 2   rating     105339 non-null  float64
 3   timestamp  105339 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.2 MB

Movies describe:


,movieId,title,genres
count,10329.000000,10329,10329
unique,NaN,10327,938
top,NaN,Men with Guns (1997),Drama
freq,NaN,2,1385
mean,31924.282893,NaN,NaN
std,37734.741149,NaN,NaN
min,1.000000,NaN,NaN
25%,3240.000000,NaN,NaN
50%,7088.000000,NaN,NaN
75%,59900.000000,NaN,NaN



Ratings describe:


,userId,movieId,rating,timestamp
count,105339.000000,105339.000000,105339.000000,1.053390e+05
mean,364.924539,13381.312477,3.516850,1.130424e+09
std,197.486905,26170.456869,1.044872,1.802660e+08
min,1.000000,1.000000,0.500000,8.285650e+08
25%,192.000000,1073.000000,3.000000,9.711008e+08
50%,383.000000,2497.000000,3.500000,1.115154e+09
75%,557.000000,5991.000000,4.000000,1.275496e+09
max,668.000000,149532.000000,5.000000,1.452405e+09


In [19]:
print("=== Missing values in Movies ===")
print(movies.isnull().sum())
print("Percentage missing:", (movies.isnull().sum() / len(movies) * 100).round(2))

print("\n=== Missing values in Ratings ===")
print(ratings.isnull().sum())
print("Percentage missing:", (ratings.isnull().sum() / len(ratings) * 100).round(2))

# Show rows that contain missing values (if any)
print("\nRows in Movies with missing values:")
display(movies[movies.isnull().any(axis=1)].head())

=== Missing values in Movies ===
movieId    0
title      0
genres     0
dtype: int64
Percentage missing: movieId    0.0
title      0.0
genres     0.0
dtype: float64

=== Missing values in Ratings ===
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Percentage missing: userId       0.0
movieId      0.0
rating       0.0
timestamp    0.0
dtype: float64

Rows in Movies with missing values:


,movieId,title,genres


In [20]:
# 1) In movies: fill missing genres with "Unknown"
movies['genres'] = movies['genres'].fillna('Unknown')

# 2) In movies: drop rows where title is missing (rarely happens)
movies = movies.dropna(subset=['title'])

# 3) In ratings: drop any row where userId, movieId, or rating is missing
ratings = ratings.dropna(subset=['userId', 'movieId', 'rating'])

# 4) Ensure correct data types
movies['movieId'] = movies['movieId'].astype(int)
ratings['userId'] = ratings['userId'].astype(int)
ratings['movieId'] = ratings['movieId'].astype(int)
ratings['rating'] = ratings['rating'].astype(float)

print("Missing values handled.")

Missing values handled.


In [21]:
print("Duplicates in Movies:", movies.duplicated().sum())
print("Duplicates in Ratings:", ratings.duplicated().sum())

# Duplicates based on movieId only
print("Duplicate movieId in Movies:", movies['movieId'].duplicated().sum())

# Duplicates based on (userId, movieId) — the important one in ratings
print("Duplicate (userId, movieId) in Ratings:",
      ratings.duplicated(subset=['userId', 'movieId']).sum())

# Show the duplicates (if any)
dups = ratings[ratings.duplicated(subset=['userId', 'movieId'], keep=False)]
print("\nExamples of duplicates:")
display(dups.sort_values(['userId', 'movieId']).head(10))

Duplicates in Movies: 0
Duplicates in Ratings: 0
Duplicate movieId in Movies: 0
Duplicate (userId, movieId) in Ratings: 0

Examples of duplicates:


,userId,movieId,rating,timestamp


In [22]:
# Remove exact (perfect) duplicates
movies = movies.drop_duplicates()
ratings = ratings.drop_duplicates()

# If the same user rated the same movie more than once → keep the latest rating
ratings = (ratings
           .sort_values('timestamp')
           .drop_duplicates(subset=['userId', 'movieId'], keep='last')
           .reset_index(drop=True))

print("After cleaning:")
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

After cleaning:
Movies: (10329, 3)
Ratings: (105339, 4)


In [23]:
print("Min rating:", ratings['rating'].min())
print("Max rating:", ratings['rating'].max())
print("\nUnique rating values:", sorted(ratings['rating'].unique()))

# Allowed values only: 0.5, 1.0, 1.5, ..., 5.0
valid_ratings = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
outliers_rating = ratings[~ratings['rating'].isin(valid_ratings)]
print("\nInvalid rating values:", len(outliers_rating))
display(outliers_rating.head())

Min rating: 0.5
Max rating: 5.0

Unique rating values: [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]

Invalid rating values: 0


,userId,movieId,rating,timestamp


In [24]:
# Convert timestamp to datetime
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')

print("Oldest rating:", ratings['date'].min())
print("Newest rating:", ratings['date'].max())

Oldest rating: 1996-04-03 20:55:54
Newest rating: 2016-01-10 05:48:39


In [25]:
today = pd.Timestamp.today()
ratings = ratings[(ratings['date'] >= '1995-01-01') & (ratings['date'] <= today)]
ratings = ratings.drop(columns=['date'])  # drop the temporary column

In [26]:
import numpy as np

# IQR method on rating (discrete values — for illustration only)
Q1 = ratings['rating'].quantile(0.25)
Q3 = ratings['rating'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")

# Z-score
from scipy import stats
z = np.abs(stats.zscore(ratings['rating']))
print("Number of values with |Z|>3:", (z > 3).sum())

IQR bounds: [1.50, 5.50]
Number of values with |Z|>3: 0


In [27]:
# Number of ratings per movie
movie_counts = ratings.groupby('movieId').size()
print(movie_counts.describe())

# Number of ratings per user
user_counts = ratings.groupby('userId').size()
print(user_counts.describe())

# Users with very few ratings (e.g., fewer than 5)
low_activity = user_counts[user_counts < 5]
print(f"Users with fewer than 5 ratings: {len(low_activity)}")

count    10325.000000
mean        10.202324
std         22.832341
min          1.000000
25%          1.000000
50%          3.000000
75%          8.000000
max        325.000000
dtype: float64
count     668.000000
mean      157.693114
std       319.712512
min        20.000000
25%        35.000000
50%        70.500000
75%       153.000000
max      5678.000000
dtype: float64
Users with fewer than 5 ratings: 0


In [28]:
MIN_RATINGS = 5
active_users = user_counts[user_counts >= MIN_RATINGS].index
ratings = ratings[ratings['userId'].isin(active_users)]

In [29]:
# 6.1 Keep only ratings whose movieId exists in movies
ratings = ratings[ratings['movieId'].isin(movies['movieId'])]

# 6.2 Drop movies that have no ratings (optional — depends on goal)
rated_movies = ratings['movieId'].unique()
movies = movies[movies['movieId'].isin(rated_movies)]

# 6.3 Filter genres — remove "Unknown" if desired
movies = movies[movies['genres'] != 'Unknown']

# 6.4 Filter active users (MIN 5 ratings)
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts >= 5].index
ratings = ratings[ratings['userId'].isin(active_users)]

# 6.5 Filter movies with at least 5 ratings
movie_counts = ratings.groupby('movieId').size()
popular_movies = movie_counts[movie_counts >= 5].index
ratings = ratings[ratings['movieId'].isin(popular_movies)]
movies = movies[movies['movieId'].isin(popular_movies)]

# 6.6 Reset index
movies = movies.reset_index(drop=True)
ratings = ratings.reset_index(drop=True)

print("=== Final shape ===")
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Unique users:", ratings['userId'].nunique())
print("Unique rated movies:", ratings['movieId'].nunique())

=== Final shape ===
Movies: (3855, 3)
Ratings: (94121, 4)
Unique users: 668
Unique rated movies: 3855


In [30]:
# 7.1 Extract release year from title
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)$').astype(float)

# 7.2 First genre (main category)
movies['main_genre'] = movies['genres'].str.split('|').str[0]

# 7.3 Number of genres
movies['num_genres'] = movies['genres'].str.count(r'\|') + 1

# 7.4 Average rating and number of ratings per movie
movie_stats = ratings.groupby('movieId').agg(
    avg_rating=('rating', 'mean'),
    num_ratings=('rating', 'count')
).reset_index()

movies = movies.merge(movie_stats, on='movieId', how='left')

print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres    year main_genre  num_genres  \
0  Adventure|Animation|Children|Comedy|Fantasy  1995.0  Adventure           5   
1                   Adventure|Children|Fantasy  1995.0  Adventure           3   
2                               Comedy|Romance  1995.0     Comedy           2   
3                         Comedy|Drama|Romance  1995.0     Comedy           3   
4                                       Comedy  1995.0     Comedy           1   

   avg_rating  num_ratings  
0    3.907328          232  
1    3.353261           92  
2    3.189655           58  
3    2.818182           11  
4    3.250000           62  


In [31]:
import pandas as pd
import numpy as np
import os

# ---------- 1. Load ----------
print("Files in raw folder:", os.listdir('../data/raw/'))
movies = pd.read_csv('../data/raw/movies.csv')
ratings = pd.read_csv('../data/raw/ratings.csv')

print("\n--- Movies Data ---"); display(movies.head())
print("\n--- Ratings Data ---"); display(ratings.head())

# ---------- 2. Info & Missing ----------
print("\nMovies missing:\n", movies.isnull().sum())
print("Ratings missing:\n", ratings.isnull().sum())

movies['genres'] = movies['genres'].fillna('Unknown')
movies = movies.dropna(subset=['title'])
ratings = ratings.dropna(subset=['userId', 'movieId', 'rating'])

# ---------- 3. Types ----------
movies['movieId'] = movies['movieId'].astype(int)
ratings['userId'] = ratings['userId'].astype(int)
ratings['movieId'] = ratings['movieId'].astype(int)
ratings['rating'] = ratings['rating'].astype(float)

# ---------- 4. Duplicates ----------
print("Dups movies:", movies.duplicated().sum())
print("Dups ratings:", ratings.duplicated(subset=['userId','movieId']).sum())

movies = movies.drop_duplicates()
ratings = (ratings.sort_values('timestamp')
                  .drop_duplicates(subset=['userId','movieId'], keep='last')
                  .reset_index(drop=True))

# ---------- 5. Outliers in rating ----------
valid_ratings = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
ratings = ratings[ratings['rating'].isin(valid_ratings)]

# ---------- 6. Timestamp check ----------
ratings['date'] = pd.to_datetime(ratings['timestamp'], unit='s')
today = pd.Timestamp.today()
ratings = ratings[(ratings['date'] >= '1995-01-01') & (ratings['date'] <= today)]
ratings = ratings.drop(columns=['date'])

# ---------- 7. Filtering ----------
ratings = ratings[ratings['movieId'].isin(movies['movieId'])]
user_counts  = ratings.groupby('userId').size()
movie_counts = ratings.groupby('movieId').size()

ratings = ratings[ratings['userId'].isin(user_counts[user_counts >= 5].index)]
ratings = ratings[ratings['movieId'].isin(movie_counts[movie_counts >= 5].index)]

movies = movies[movies['movieId'].isin(ratings['movieId'].unique())]
movies = movies.reset_index(drop=True)
ratings = ratings.reset_index(drop=True)

# ---------- 8. Feature Engineering ----------
movies['year'] = movies['title'].str.extract(r'\((\d{4})\)$').astype(float)
movies['num_genres'] = movies['genres'].str.count(r'\|') + 1

movie_stats = ratings.groupby('movieId').agg(
    avg_rating=('rating','mean'),
    num_ratings=('rating','count')
).reset_index()
movies = movies.merge(movie_stats, on='movieId', how='left')

# ---------- 9. Final check ----------
print("\n=== Final shapes ===")
print("Movies :", movies.shape)
print("Ratings:", ratings.shape)
print("Users  :", ratings['userId'].nunique())
print("Items  :", ratings['movieId'].nunique())

movies.to_csv('../data/processed/movies_clean.csv', index=False)
ratings.to_csv('../data/processed/ratings_clean.csv', index=False)

Files in raw folder: ['movies.csv', 'ratings.csv']

--- Movies Data ---


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



--- Ratings Data ---


,userId,movieId,rating,timestamp
0,1,16,4.0,1217897793
1,1,24,1.5,1217895807
2,1,32,4.0,1217896246
3,1,47,4.0,1217896556
4,1,50,4.0,1217896523



Movies missing:
 movieId    0
title      0
genres     0
dtype: int64
Ratings missing:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
Dups movies: 0
Dups ratings: 0

=== Final shapes ===
Movies : (3855, 7)
Ratings: (94121, 4)
Users  : 668
Items  : 3855
